# Phase 1 Review Notebook Index

The former combined Phase 1 review has been split into two focused, non-mutating-by-default notebooks:

1. [01a_phase_1_source_registry_review.ipynb](01a_phase_1_source_registry_review.ipynb) — Phase 1A source registry inputs, provenance, quality checks, reference sources, and Phase 1B readiness.
2. [01b_phase_1_provider_profiling_review.ipynb](01b_phase_1_provider_profiling_review.ipynb) — Phase 1B provider profiles, strategies, candidate links, debug evidence, unresolved providers, and Phase 1.5/Phase 2 readiness.

Neither notebook implements Phase 1.5 or Phase 2. Live samples and optional exports are disabled by default.

## Expected artifact availability

In [1]:
from pathlib import Path
import json
import subprocess
import sys
from urllib.parse import urlparse

import pandas as pd
from IPython.display import Image, display


def find_repo_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'mutual_fund_ingestion' / '__init__.py').is_file():
            return candidate
    raise RuntimeError('Run this notebook from inside the Financial Analytics Work repository.')


def available_columns(frame, requested):
    return [column for column in requested if column in frame.columns]


def show_columns(frame, requested, *, empty_message='No matching records.'):
    columns = available_columns(frame, requested)
    if frame.empty:
        print(empty_message)
        return frame.reindex(columns=columns)
    return frame.loc[:, columns]


def read_jsonl(path):
    if not path.exists():
        print(f'Missing optional artifact: {path}')
        return []
    records = []
    for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), 1):
        if line.strip():
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                print(f'Skipping invalid JSONL record {path}:{line_number}: {exc}')
    return records

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ROOT

EXPECTED = {
    'source_registry_config': ROOT / 'configs/amc_sources.yaml',
    'source_registry_candidates': ROOT / 'data/raw/mutual_funds/source_registry/source_registry_candidates.jsonl',
    'source_registry_latest': ROOT / 'data/raw/mutual_funds/source_registry/source_registry.latest.json',
    'source_registry_report': ROOT / 'data/reports/mutual_funds/source_registry_report.html',
    'provider_profile_history': ROOT / 'data/raw/mutual_funds/provider_profiles/provider_profiles.jsonl',
    'provider_profiles_latest': ROOT / 'data/raw/mutual_funds/provider_profiles/provider_profiles.latest.json',
    'provider_profile_summary': ROOT / 'data/reports/mutual_funds/provider_profile_summary.csv',
    'provider_profile_report': ROOT / 'data/reports/mutual_funds/provider_profile_report.html',
    'provider_debug_root': ROOT / 'data/debug/mutual_funds/provider_profiles',
}
pd.DataFrame([{'artifact': name, 'path': str(path), 'exists': path.exists()} for name, path in EXPECTED.items()])

,artifact,path,exists
0,source_registry_config,/Users/vedaangchopra/all_data/complete_technic...,True
1,source_registry_candidates,/Users/vedaangchopra/all_data/complete_technic...,True
2,source_registry_latest,/Users/vedaangchopra/all_data/complete_technic...,True
3,source_registry_report,/Users/vedaangchopra/all_data/complete_technic...,False
4,provider_profile_history,/Users/vedaangchopra/all_data/complete_technic...,True
5,provider_profiles_latest,/Users/vedaangchopra/all_data/complete_technic...,True
6,provider_profile_summary,/Users/vedaangchopra/all_data/complete_technic...,True
7,provider_profile_report,/Users/vedaangchopra/all_data/complete_technic...,True
8,provider_debug_root,/Users/vedaangchopra/all_data/complete_technic...,True


## Usage

Open **01A** first to validate that the source registry is suitable for provider profiling. Then open **01B** to inspect saved provider strategies and identify unresolved providers for a future Phase 1.5 design.